## TEST BUCKETING PREDICTION MODELS

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Load data
df_post = pd.read_csv("../data_final/eurogate_postdeparture.csv")

# Create target bucket
# 0 = short dwell, 1 = long dwell
df_post["dwell_bucket"] = (
    df_post["dwell_hours"] >= 240
).astype(int)

print("Bucket distribution:")
print(df_post["dwell_bucket"].value_counts(normalize=True))

# Drop leakage / non-model columns
drop_cols = [
    "dwell_hours",          # used to create target
    "dwell_bucket",         # target itself
    "snapshot_time",
    "snapshot_date",
    "snapshot_year",
    "snapshot_hour",
    "snapshot_minute",
    "snapshot_second",
    "arrival_year",
    "departureTime",
    "remaining_dwell_hours",
    "elapsed_dwell_hours",
    "Unnamed: 0",
    "Unnamed: 0.1",
    "Unnamed: 0.2"
]

df_post = df_post.dropna(subset=["dwell_bucket"]).copy()

X = df_post.drop(columns=[c for c in drop_cols if c in df_post.columns])
y = df_post["dwell_bucket"]

print("Target leakage check:")
print("dwell_bucket in X?", "dwell_bucket" in X.columns)
print("dwell_hours in X?", "dwell_hours" in X.columns)

# Clean features
X = X.replace([np.inf, -np.inf], np.nan)

object_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Dropping object columns:", object_cols)
X = X.drop(columns=object_cols)

bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

X = X.fillna(X.median(numeric_only=True))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Evaluation function
def evaluate_classifier(name, model):
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, prob)
    else:
        prob = None
        auc = np.nan

    acc = accuracy_score(y_test, pred)
    bal_acc = balanced_accuracy_score(y_test, pred)

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)
    print(f"Accuracy: {acc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"ROC AUC: {auc:.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, pred))

    print("\nClassification report:")
    print(classification_report(
        y_test,
        pred,
        target_names=["short_<240h", "long_240h+"]
    ))

    return {
        "model_name": name,
        "model": model,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "roc_auc": auc,
        "pred": pred,
        "prob": prob
    }

# Models
models = [
    ("Baseline Most Frequent", DummyClassifier(strategy="most_frequent")),

    ("Extra Trees Balanced", ExtraTreesClassifier(
        n_estimators=500,
        max_depth=40,
        min_samples_leaf=2,
        min_samples_split=5,
        max_features=0.75,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),

    ("Random Forest Balanced", RandomForestClassifier(
        n_estimators=500,
        max_depth=40,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
]

results = []

for name, model in models:
    results.append(evaluate_classifier(name, model))

# Summary
summary = pd.DataFrame([
    {
        "model": r["model_name"],
        "accuracy": r["accuracy"],
        "balanced_accuracy": r["balanced_accuracy"],
        "roc_auc": r["roc_auc"]
    }
    for r in results
]).sort_values("balanced_accuracy", ascending=False)

print("\nModel comparison:")
print(summary)

summary.to_csv(
    "../feature_importance/dwell_bucket_model_comparison.csv",
    index=False
)

# Feature importance for best tree model
best = max(results, key=lambda r: r["balanced_accuracy"])
best_model = best["model"]

if hasattr(best_model, "feature_importances_"):
    importance = pd.DataFrame({
        "feature": X.columns,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\nTop 30 Features:")
    print(importance.head(30))

    importance.to_csv(
        "../feature_importance/dwell_bucket_feature_importance.csv",
        index=False
    )

# Save predictions
predictions = pd.DataFrame({
    "actual_bucket": y_test,
    "predicted_bucket": best["pred"],
    "predicted_probability_long": best["prob"]
})

predictions.to_csv(
    "../feature_importance/dwell_bucket_predictions.csv",
    index=False
)

print("\nSaved:")
print("../feature_importance/dwell_bucket_model_comparison.csv")
print("../feature_importance/dwell_bucket_feature_importance.csv")
print("../feature_importance/dwell_bucket_predictions.csv")

Bucket distribution:
dwell_bucket
0    0.904301
1    0.095699
Name: proportion, dtype: float64
Target leakage check:
dwell_bucket in X? False
dwell_hours in X? False
Dropping object columns: []

Baseline Most Frequent
Accuracy: 0.9043
Balanced accuracy: 0.5000
ROC AUC: 0.5000

Confusion matrix:
[[26239     0]
 [ 2777     0]]

Classification report:
              precision    recall  f1-score   support

 short_<240h       0.90      1.00      0.95     26239
  long_240h+       0.00      0.00      0.00      2777

    accuracy                           0.90     29016
   macro avg       0.45      0.50      0.47     29016
weighted avg       0.82      0.90      0.86     29016



/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Extra Trees Balanced
Accuracy: 0.9302
Balanced accuracy: 0.8272
ROC AUC: 0.9265

Confusion matrix:
[[25049  1190]
 [  834  1943]]

Classification report:
              precision    recall  f1-score   support

 short_<240h       0.97      0.95      0.96     26239
  long_240h+       0.62      0.70      0.66      2777

    accuracy                           0.93     29016
   macro avg       0.79      0.83      0.81     29016
weighted avg       0.93      0.93      0.93     29016


Random Forest Balanced
Accuracy: 0.9412
Balanced accuracy: 0.8179
ROC AUC: 0.9258

Confusion matrix:
[[25462   777]
 [  929  1848]]

Classification report:
              precision    recall  f1-score   support

 short_<240h       0.96      0.97      0.97     26239
  long_240h+       0.70      0.67      0.68      2777

    accuracy                           0.94     29016
   macro avg       0.83      0.82      0.83     29016
weighted avg       0.94      0.94      0.94     29016


Model comparison:
               

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# Load data
df_post = pd.read_csv("../data_final/eurogate_postdeparture.csv")

# Create binary bucket target
# 0 = short dwell: < 240 hours
# 1 = long dwell: >= 240 hours
df_post["dwell_bucket"] = (df_post["dwell_hours"] >= 240).astype(int)

print("Bucket distribution:")
print(df_post["dwell_bucket"].value_counts(normalize=True))

# Useful quick operational checks
print("\nLong-stay rate by import/export:")
print(df_post.groupby("iedCode")["dwell_bucket"].mean())

print("\nDwell hours by import/export:")
print(df_post.groupby("iedCode")["dwell_hours"].describe())

# Drop leakage / non-model columns
drop_cols = [
    "dwell_hours",
    "dwell_bucket",
    "snapshot_time",
    "snapshot_date",
    "snapshot_year",
    "snapshot_hour",
    "snapshot_minute",
    "snapshot_second",
    "arrival_year",
    "departureTime",
    "remaining_dwell_hours",
    "elapsed_dwell_hours",
    "Unnamed: 0",
    "Unnamed: 0.1",
    "Unnamed: 0.2"
]

X = df_post.drop(columns=[c for c in drop_cols if c in df_post.columns])
y = df_post["dwell_bucket"]

print("\nTarget leakage check:")
print("dwell_bucket in X?", "dwell_bucket" in X.columns)
print("dwell_hours in X?", "dwell_hours" in X.columns)

# Clean features
X = X.replace([np.inf, -np.inf], np.nan)

object_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("\nDropping object columns:", object_cols)
X = X.drop(columns=object_cols)

bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

X = X.fillna(X.median(numeric_only=True))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Evaluation helpers
def evaluate_at_threshold(name, y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)

    return {
        "model": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_long": precision_score(y_true, pred, zero_division=0),
        "recall_long": recall_score(y_true, pred, zero_division=0),
        "f1_long": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probs),
        "predicted_long_rate": pred.mean(),
        "actual_long_rate_among_predicted_long": (
            y_true[pred == 1].mean() if pred.sum() > 0 else np.nan
        )
    }


def print_model_report(name, model):
    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]

    # default threshold
    pred_default = (probs >= 0.50).astype(int)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("\nDefault threshold = 0.50")
    print(f"Accuracy: {accuracy_score(y_test, pred_default):.4f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_test, pred_default):.4f}")
    print(f"ROC AUC: {roc_auc_score(y_test, probs):.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, pred_default))

    print("\nClassification report:")
    print(classification_report(
        y_test,
        pred_default,
        target_names=["short_<240h", "long_240h+"],
        zero_division=0
    ))

    # Test multiple thresholds
    thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

    threshold_results = pd.DataFrame([
        evaluate_at_threshold(name, y_test, probs, t)
        for t in thresholds
    ])

    print("\nThreshold comparison:")
    print(threshold_results.sort_values("threshold"))

    # False negatives at default threshold
    results = X_test.copy()
    results["actual_bucket"] = y_test.values
    results["predicted_bucket"] = pred_default
    results["probability_long"] = probs

    false_negatives = results[
        (results["actual_bucket"] == 1) &
        (results["predicted_bucket"] == 0)
    ].copy()

    false_positives = results[
        (results["actual_bucket"] == 0) &
        (results["predicted_bucket"] == 1)
    ].copy()

    print("\nFalse negatives:", len(false_negatives))
    print("False positives:", len(false_positives))

    return model, probs, threshold_results, results, false_negatives, false_positives

# Models
models = [
    ("Baseline Most Frequent", DummyClassifier(strategy="most_frequent")),

    ("Extra Trees Balanced", ExtraTreesClassifier(
        n_estimators=500,
        max_depth=40,
        min_samples_leaf=2,
        min_samples_split=5,
        max_features=0.75,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),

    ("Random Forest Balanced", RandomForestClassifier(
        n_estimators=500,
        max_depth=40,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
]

all_thresholds = []
model_outputs = {}

for name, model in models:
    if name.startswith("Baseline"):
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        print("\n" + "=" * 60)
        print(name)
        print("=" * 60)
        print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
        print(f"Balanced accuracy: {balanced_accuracy_score(y_test, pred):.4f}")
        print("\nConfusion matrix:")
        print(confusion_matrix(y_test, pred))
        print("\nClassification report:")
        print(classification_report(
            y_test,
            pred,
            target_names=["short_<240h", "long_240h+"],
            zero_division=0
        ))

    else:
        fitted_model, probs, threshold_df, result_df, fn_df, fp_df = print_model_report(
            name,
            model
        )

        all_thresholds.append(threshold_df)

        model_outputs[name] = {
            "model": fitted_model,
            "probs": probs,
            "results": result_df,
            "false_negatives": fn_df,
            "false_positives": fp_df
        }

# Save threshold comparison
threshold_summary = pd.concat(all_thresholds, ignore_index=True)

threshold_summary.to_csv(
    "../feature_importance/dwell_bucket_threshold_comparison.csv",
    index=False
)

print("\nSaved threshold comparison:")
print("../feature_importance/dwell_bucket_threshold_comparison.csv")

# Choose best model by balanced accuracy at tested thresholds
best_row = threshold_summary.sort_values(
    ["balanced_accuracy", "recall_long"],
    ascending=False
).iloc[0]

best_name = best_row["model"]
best_threshold = best_row["threshold"]

print("\nBest threshold result:")
print(best_row)

best_info = model_outputs[best_name]
best_model = best_info["model"]
best_probs = best_info["probs"]
best_pred = (best_probs >= best_threshold).astype(int)

# Save final predictions
predictions = pd.DataFrame({
    "actual_bucket": y_test.values,
    "predicted_bucket": best_pred,
    "probability_long": best_probs,
    "threshold_used": best_threshold
})

predictions.to_csv(
    "../feature_importance/dwell_bucket_predictions_threshold_tuned.csv",
    index=False
)

# Save false negatives / false positives at best threshold
error_analysis = X_test.copy()
error_analysis["actual_bucket"] = y_test.values
error_analysis["predicted_bucket"] = best_pred
error_analysis["probability_long"] = best_probs

false_negatives_best = error_analysis[
    (error_analysis["actual_bucket"] == 1) &
    (error_analysis["predicted_bucket"] == 0)
].copy()

false_positives_best = error_analysis[
    (error_analysis["actual_bucket"] == 0) &
    (error_analysis["predicted_bucket"] == 1)
].copy()

false_negatives_best.to_csv(
    "../feature_importance/dwell_bucket_false_negatives.csv",
    index=False
)

false_positives_best.to_csv(
    "../feature_importance/dwell_bucket_false_positives.csv",
    index=False
)

# Feature importance for best model
if hasattr(best_model, "feature_importances_"):
    importance = pd.DataFrame({
        "feature": X.columns,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\nTop 30 Features:")
    print(importance.head(30))

    importance.to_csv(
        "../feature_importance/dwell_bucket_feature_importance_threshold_tuned.csv",
        index=False
    )

# Optional: save model
import joblib

joblib.dump(
    best_model,
    "../feature_importance/dwell_bucket_best_model.joblib"
)

print("\nSaved:")
print("../feature_importance/dwell_bucket_threshold_comparison.csv")
print("../feature_importance/dwell_bucket_predictions_threshold_tuned.csv")
print("../feature_importance/dwell_bucket_false_negatives.csv")
print("../feature_importance/dwell_bucket_false_positives.csv")
print("../feature_importance/dwell_bucket_feature_importance_threshold_tuned.csv")
print("../feature_importance/dwell_bucket_best_model.joblib")

Bucket distribution:
dwell_bucket
0    0.904301
1    0.095699
Name: proportion, dtype: float64

Long-stay rate by import/export:
iedCode
0    0.055633
1    0.184872
Name: dwell_bucket, dtype: float64

Dwell hours by import/export:
            count        mean         std       min        25%         50%  \
iedCode                                                                      
0        100103.0   91.771561   95.824826  0.982222  37.689722   67.673889   
1         44977.0  153.953748  137.078794  1.028611  65.784444  121.435833   

                75%          max  
iedCode                           
0        114.192222  3490.152222  
1        202.388889  4986.133333  

Target leakage check:
dwell_bucket in X? False
dwell_hours in X? False

Dropping object columns: []

Baseline Most Frequent
Accuracy: 0.9043
Balanced accuracy: 0.5000

Confusion matrix:
[[26239     0]
 [ 2777     0]]

Classification report:
              precision    recall  f1-score   support

 short_<240h       

In [8]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Load data
df_post = pd.read_csv("../data_final/eurogate_postdeparture.csv")

# Create 3 dwell buckets
# 0 = short: 0-72 hours
# 1 = medium: 72-240 hours
# 2 = long: 240+ hours
df_post["dwell_bucket_3"] = pd.cut(
    df_post["dwell_hours"],
    bins=[0, 72, 240, np.inf],
    labels=[0, 1, 2],
    include_lowest=True
).astype(int)

print("3-bucket distribution:")
print(df_post["dwell_bucket_3"].value_counts(normalize=True).sort_index())

# Drop leakage / non-model columns
drop_cols = [
    "dwell_hours",
    "dwell_bucket",
    "dwell_bucket_3",
    "snapshot_time",
    "snapshot_date",
    "snapshot_year",
    "snapshot_hour",
    "snapshot_minute",
    "snapshot_second",
    "arrival_year",
    "departureTime",
    "remaining_dwell_hours",
    "elapsed_dwell_hours",
    "Unnamed: 0",
    "Unnamed: 0.1",
    "Unnamed: 0.2"
]

X = df_post.drop(columns=[c for c in drop_cols if c in df_post.columns])
y = df_post["dwell_bucket_3"]

# Clean features
X = X.replace([np.inf, -np.inf], np.nan)

object_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("\nDropping object columns:", object_cols)
X = X.drop(columns=object_cols)

bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

X = X.fillna(X.median(numeric_only=True))

print("\nLeakage check:")
print("dwell_hours in X?", "dwell_hours" in X.columns)
print("dwell_bucket_3 in X?", "dwell_bucket_3" in X.columns)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Evaluation function
def evaluate_multiclass(name, model):
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)

    acc = accuracy_score(y_test, pred)
    bal_acc = balanced_accuracy_score(y_test, pred)

    try:
        auc = roc_auc_score(
            y_test,
            prob,
            multi_class="ovr",
            average="macro"
        )
    except Exception:
        auc = np.nan

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(f"Accuracy: {acc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"Macro ROC AUC: {auc:.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, pred))

    print("\nClassification report:")
    print(classification_report(
        y_test,
        pred,
        target_names=[
            "short_0_72h",
            "medium_72_240h",
            "long_240h_plus"
        ],
        zero_division=0
    ))

    return {
        "model_name": name,
        "model": model,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_roc_auc": auc,
        "pred": pred,
        "prob": prob
    }

# Two best model families from 2-bucket test
models = [
    ("Extra Trees Balanced", ExtraTreesClassifier(
        n_estimators=500,
        max_depth=40,
        min_samples_leaf=2,
        min_samples_split=5,
        max_features=0.75,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),

    ("Random Forest Balanced", RandomForestClassifier(
        n_estimators=500,
        max_depth=40,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
]

results = []

for name, model in models:
    results.append(evaluate_multiclass(name, model))

# Model comparison
summary = pd.DataFrame([
    {
        "model": r["model_name"],
        "accuracy": r["accuracy"],
        "balanced_accuracy": r["balanced_accuracy"],
        "macro_roc_auc": r["macro_roc_auc"]
    }
    for r in results
]).sort_values("balanced_accuracy", ascending=False)

print("\n3-bucket model comparison:")
print(summary)

summary.to_csv(
    "../feature_importance/dwell_3bucket_model_comparison.csv",
    index=False
)

# Save best model predictions
best = max(results, key=lambda r: r["balanced_accuracy"])

predictions = pd.DataFrame({
    "actual_bucket": y_test.values,
    "predicted_bucket": best["pred"],
    "prob_short_0_72h": best["prob"][:, 0],
    "prob_medium_72_240h": best["prob"][:, 1],
    "prob_long_240h_plus": best["prob"][:, 2]
})

predictions.to_csv(
    "../feature_importance/dwell_3bucket_predictions.csv",
    index=False
)

# Feature importance
best_model = best["model"]

if hasattr(best_model, "feature_importances_"):
    importance = pd.DataFrame({
        "feature": X.columns,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\nTop 30 Features:")
    print(importance.head(30))

    importance.to_csv(
        "../feature_importance/dwell_3bucket_feature_importance.csv",
        index=False
    )

# Save model
joblib.dump(
    best_model,
    "../feature_importance/dwell_3bucket_best_model.joblib"
)

print("\nSaved:")
print("../feature_importance/dwell_3bucket_model_comparison.csv")
print("../feature_importance/dwell_3bucket_predictions.csv")
print("../feature_importance/dwell_3bucket_feature_importance.csv")
print("../feature_importance/dwell_3bucket_best_model.joblib")

3-bucket distribution:
dwell_bucket_3
0    0.452316
1    0.451985
2    0.095699
Name: proportion, dtype: float64

Dropping object columns: []

Leakage check:
dwell_hours in X? False
dwell_bucket_3 in X? False

Extra Trees Balanced
Accuracy: 0.7462
Balanced accuracy: 0.7332
Macro ROC AUC: 0.8925

Confusion matrix:
[[10238  2491   395]
 [ 2786  9479   850]
 [  262   580  1935]]

Classification report:
                precision    recall  f1-score   support

   short_0_72h       0.77      0.78      0.78     13124
medium_72_240h       0.76      0.72      0.74     13115
long_240h_plus       0.61      0.70      0.65      2777

      accuracy                           0.75     29016
     macro avg       0.71      0.73      0.72     29016
  weighted avg       0.75      0.75      0.75     29016


Random Forest Balanced
Accuracy: 0.7550
Balanced accuracy: 0.7310
Macro ROC AUC: 0.8936

Confusion matrix:
[[10569  2317   238]
 [ 3009  9495   611]
 [  310   624  1843]]

Classification report:
      

## FINAL MODEL SCRIPT

In [ ]:
## FINAL SCRIPT
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Load data
df = pd.read_csv("../data_final/eurogate_postdeparture.csv")

# Create 3 dwell-time buckets
# 0 = short: 0-72 hours
# 1 = medium: 72-240 hours
# 2 = long: 240+ hours
df["dwell_bucket_3"] = pd.cut(
    df["dwell_hours"],
    bins=[0, 72, 240, np.inf],
    labels=[0, 1, 2],
    include_lowest=True
).astype(int)

print("3-bucket distribution:")
print(df["dwell_bucket_3"].value_counts(normalize=True).sort_index())

# Drop leakage / non-model columns
drop_cols = [
    "dwell_hours",
    "dwell_bucket",
    "dwell_bucket_3",
    "snapshot_time",
    "snapshot_date",
    "snapshot_year",
    "snapshot_hour",
    "snapshot_minute",
    "snapshot_second",
    "arrival_year",
    "departureTime",
    "remaining_dwell_hours",
    "elapsed_dwell_hours",
    "Unnamed: 0",
    "Unnamed: 0.1",
    "Unnamed: 0.2"
]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df["dwell_bucket_3"]

# Clean features
X = X.replace([np.inf, -np.inf], np.nan)

object_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Dropping object columns:", object_cols)
X = X.drop(columns=object_cols)

bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

X = X.fillna(X.median(numeric_only=True))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train Extra Trees classifier
model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=40,
    min_samples_leaf=2,
    min_samples_split=5,
    max_features=0.75,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predict / evaluate
pred = model.predict(X_test)
prob = model.predict_proba(X_test)

print("\nExtra Trees 3-Bucket Classifier")
print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, pred):.4f}")
print(f"Macro ROC AUC: {roc_auc_score(y_test, prob, multi_class='ovr', average='macro'):.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, pred))

print("\nClassification report:")
print(classification_report(
    y_test,
    pred,
    target_names=[
        "short_0_72h",
        "medium_72_240h",
        "long_240h_plus"
    ],
    zero_division=0
))

# Save predictions
predictions = pd.DataFrame({
    "actual_bucket": y_test.values,
    "predicted_bucket": pred,
    "prob_short_0_72h": prob[:, 0],
    "prob_medium_72_240h": prob[:, 1],
    "prob_long_240h_plus": prob[:, 2]
})

predictions.to_csv(
    "../feature_importance/dwell_3bucket_extra_trees_predictions.csv",
    index=False
)

# Save feature importance
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

importance.to_csv(
    "../feature_importance/dwell_3bucket_extra_trees_feature_importance.csv",
    index=False
)

# Save model
joblib.dump(
    model,
    "../feature_importance/dwell_3bucket_extra_trees_model.joblib"
)

print("\nSaved:")
print("../feature_importance/dwell_3bucket_extra_trees_predictions.csv")
print("../feature_importance/dwell_3bucket_extra_trees_feature_importance.csv")
print("../feature_importance/dwell_3bucket_extra_trees_model.joblib")

3-bucket distribution:
dwell_bucket_3
0    0.452316
1    0.451985
2    0.095699
Name: proportion, dtype: float64
Dropping object columns: []

Extra Trees 3-Bucket Classifier
Accuracy: 0.7462
Balanced accuracy: 0.7332
Macro ROC AUC: 0.8925

Confusion matrix:
[[10238  2491   395]
 [ 2786  9479   850]
 [  262   580  1935]]

Classification report:
                precision    recall  f1-score   support

   short_0_72h       0.77      0.78      0.78     13124
medium_72_240h       0.76      0.72      0.74     13115
long_240h_plus       0.61      0.70      0.65      2777

      accuracy                           0.75     29016
     macro avg       0.71      0.73      0.72     29016
  weighted avg       0.75      0.75      0.75     29016


Saved:
../feature_importance/dwell_3bucket_extra_trees_predictions.csv
../feature_importance/dwell_3bucket_extra_trees_feature_importance.csv
../feature_importance/dwell_3bucket_extra_trees_model.joblib
